In [1]:
import random, numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

seed = 42
random.seed(seed); np.random.seed(seed)
torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [2]:
class TwoCropsTransform:
    def __init__(self, base_transform):
        self.base_transform = base_transform
    def __call__(self, x):
        q = self.base_transform(x)
        k = self.base_transform(x)
        return q, k


base_transform = transforms.Compose([
    transforms.RandomResizedCrop(28, scale=(0.8, 1.0)),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
])

train_ds = datasets.MNIST(
    root="./data", train=True, download=True,
    transform=TwoCropsTransform(base_transform)
)

test_ds_plain = datasets.MNIST(
    root="./data", train=False, download=True,
    transform=transforms.ToTensor()
)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
test_loader_plain = DataLoader(test_ds_plain, batch_size=512, shuffle=False, num_workers=2, pin_memory=True)

In [16]:
class EncoderCNN(nn.Module):
    def __init__(self, emb_dim=128):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),  # 14x14
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),  # 7x7
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.fc = nn.Linear(128, emb_dim)

    def forward(self, x):
        x = self.features(x).flatten(1)
        z = self.fc(x)
        z = F.normalize(z, dim=1)
        return z

class ProjectionHead(nn.Module):
    def __init__(self, in_dim=128, proj_dim=128):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, 256),
            nn.ReLU(),
            nn.Linear(256, proj_dim)
        )

    def forward(self, x):
        x = self.mlp(x)
        x = F.normalize(x, dim=1)
        return x

class RakEmbeddingModel(nn.Module):
    def __init__(self, emb_dim=128, proj_dim=128):
        super().__init__()
        self.encoder = EncoderCNN(emb_dim=emb_dim)
        self.projector = ProjectionHead(in_dim=emb_dim, proj_dim=proj_dim)

    def forward(self, x):
        h = self.encoder(x)
        z = self.projector(h)      # for InfoNCE training
        return h, z

model = RakEmbeddingModel(emb_dim=128, proj_dim=128).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [4]:
def info_nce_loss(z1, z2, temperature=0.2):
    """
    z1, z2: (B, D) normalized
    positives: (i in z1) matches (i in z2)
    negatives: all other samples in batch
    """
    B = z1.size(0)
    z = torch.cat([z1, z2], dim=0)             # (2B, D)

    sim = torch.matmul(z, z.T)                 # (2B, 2B) cosine sim since normalized
    sim = sim / temperature

    # remove self-similarity
    mask = torch.eye(2 * B, device=z.device, dtype=torch.bool)
    sim = sim.masked_fill(mask, -1e9)

    # positive indices: for i in [0..B-1], positive is i+B; for i in [B..2B-1], positive is i-B
    pos_idx = torch.arange(2 * B, device=z.device)
    pos_idx = (pos_idx + B) % (2 * B)

    loss = F.cross_entropy(sim, pos_idx)
    return loss

In [5]:
import copy, random
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import umap
import matplotlib.pyplot as plt
import matplotlib.animation as animation

digits = [3, 7, 9]
epochs = 10
temperature = 0.2


K_PER_DIGIT = 250

fps = 2
dpi = 170
marker_size = 32
# ---------------------------

device = next(model.parameters()).device

# Pick a FIXED subset of test indices for the 3 digits (same points every epoch)
labels_all = np.array([test_ds_plain[i][1] for i in range(len(test_ds_plain))])

idxs = []
for d in digits:
    candidates = np.where(labels_all == d)[0]
    if len(candidates) < K_PER_DIGIT:
        raise ValueError(f"Not enough samples for digit {d}. Found {len(candidates)}.")
    chosen = np.random.RandomState(42 + d).choice(candidates, size=K_PER_DIGIT, replace=False)
    idxs.append(chosen)

subset_idx = np.concatenate(idxs).tolist()
subset_labels = labels_all[subset_idx].astype(int)

vis_loader = DataLoader(Subset(test_ds_plain, subset_idx), batch_size=512, shuffle=False)

# saving encoder checkpoints every epoch
checkpoints = []

def train_one_epoch(model, loader, temperature=0.2):
    model.train()
    total = 0.0
    for (x1, x2), _ in loader:
        x1 = x1.to(device)
        x2 = x2.to(device)

        optimizer.zero_grad()
        _, z1 = model(x1)
        _, z2 = model(x2)
        loss = info_nce_loss(z1, z2, temperature=temperature)
        loss.backward()
        optimizer.step()

        total += loss.item() * x1.size(0)
    return total / len(loader.dataset)

@torch.no_grad()
def compute_subset_embeddings(encoder, state_dict, loader):
    encoder.load_state_dict(state_dict)
    encoder.eval()
    embs = []
    for x, _ in loader:
        x = x.to(device)
        h = encoder(x)          # (B, D) normalized
        embs.append(h.cpu())
    return torch.cat(embs, dim=0).numpy()

losses = []
for ep in range(1, epochs + 1):
    l = train_one_epoch(model, train_loader, temperature=temperature)
    losses.append(l)
    checkpoints.append(copy.deepcopy(model.encoder.state_dict()))
    print(f"Epoch {ep}/{epochs} | InfoNCE loss={l:.4f}")

# Compute embeddings for the same points at each epoch
embs_per_epoch = []
for sd in checkpoints:
    E = compute_subset_embeddings(model.encoder, sd, vis_loader)  # (3*K, D)
    embs_per_epoch.append(E)

print("Frames:", len(embs_per_epoch), " Embeddings shape:", embs_per_epoch[0].shape)

all_embs = np.concatenate(embs_per_epoch, axis=0)  # (epochs*3K, D)

reducer = umap.UMAP(
    n_components=2, n_neighbors=15, min_dist=0.1,
    metric="cosine", random_state=42
)
all_proj = reducer.fit_transform(all_embs)  # (epochs*3K, 2)

E = len(embs_per_epoch)
N = embs_per_epoch[0].shape[0]
proj_per_epoch = [all_proj[i*N:(i+1)*N] for i in range(E)]


colors = {
    digits[0]: np.array([0.90, 0.10, 0.10, 1.0]),  # red
    digits[1]: np.array([0.10, 0.65, 0.10, 1.0]),  # green
    digits[2]: np.array([0.10, 0.25, 0.95, 1.0]),  # blue
}


subset_labels = subset_labels.astype(int)
idx_by_digit = {d: np.where(subset_labels == d)[0] for d in digits}

all_pts = np.concatenate(proj_per_epoch, axis=0)
x_min, x_max = all_pts[:,0].min(), all_pts[:,0].max()
y_min, y_max = all_pts[:,1].min(), all_pts[:,1].max()
pad_x = 0.08 * (x_max - x_min + 1e-9)
pad_y = 0.08 * (y_max - y_min + 1e-9)

fig, ax = plt.subplots(figsize=(7, 7), dpi=dpi)
fig.patch.set_facecolor("white")
ax.set_facecolor("white")
ax.set_xlim(x_min - pad_x, x_max + pad_x)
ax.set_ylim(y_min - pad_y, y_max + pad_y)
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")

scatters = []
for d in digits:
    sc = ax.scatter([], [], s=marker_size, linewidths=0, alpha=0.95)
    scatters.append(sc)


for d in digits:
    ax.scatter([], [], s=90, c=colors[d].reshape(1,4), label=f"Digit {d}", linewidths=0)
ax.legend(loc="upper right", frameon=True)

title = ax.text(0.5, 1.02, "", transform=ax.transAxes, ha="center", va="bottom", fontsize=13)

def init():
    for sc in scatters:
        sc.set_offsets(np.zeros((0, 2)))
    title.set_text("UMAP Embeddings | Epoch 1")
    return (*scatters, title)

def update(frame):
    coords = proj_per_epoch[frame]
    for k, d in enumerate(digits):
        pts = coords[idx_by_digit[d]]
        scatters[k].set_offsets(pts)
        scatters[k].set_facecolors(np.tile(colors[d], (pts.shape[0], 1)))
    title.set_text(f"UMAP Embeddings | Epoch {frame+1}/{E} | Digits: {digits}")
    return (*scatters, title)

ani = animation.FuncAnimation(
    fig, update, frames=E, init_func=init,
    interval=int(1000 / fps),
    blit=False
)

mp4_path = f"umap_digits_{digits[0]}_{digits[1]}_{digits[2]}_evolution.mp4"
writer = animation.FFMpegWriter(fps=fps, bitrate=3500)
ani.save(mp4_path, writer=writer)

plt.close(fig)
print("Saved:", mp4_path)

Epoch 1/10 | InfoNCE loss=3.8300
Epoch 2/10 | InfoNCE loss=2.5495
Epoch 3/10 | InfoNCE loss=2.3262
Epoch 4/10 | InfoNCE loss=2.2190
Epoch 5/10 | InfoNCE loss=2.1508
Epoch 6/10 | InfoNCE loss=2.1040
Epoch 7/10 | InfoNCE loss=2.0666
Epoch 8/10 | InfoNCE loss=2.0401
Epoch 9/10 | InfoNCE loss=2.0126
Epoch 10/10 | InfoNCE loss=1.9964
Frames: 10  Embeddings shape: (750, 128)


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Saved: umap_digits_3_7_9_evolution.mp4


In [12]:
@torch.no_grad()
def extract_embeddings(encoder, loader, device=None):
    encoder.eval()
    if device is None:
        device = next(encoder.parameters()).device

    all_embs = []
    all_labels = []

    for x, y in loader:
        x = x.to(device)
        h = encoder(x)
        all_embs.append(h.cpu())
        all_labels.append(y.cpu())

    return torch.cat(all_embs), torch.cat(all_labels)

In [6]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

def knn_accuracy(train_embs, train_labels, test_embs, test_labels, k=10):
    knn = KNeighborsClassifier(n_neighbors=k, metric="cosine")
    knn.fit(train_embs.numpy(), train_labels.numpy())
    preds = knn.predict(test_embs.numpy())
    return accuracy_score(test_labels.numpy(), preds)

In [7]:
from sklearn.linear_model import LogisticRegression

def linear_probe_accuracy(train_embs, train_labels, test_embs, test_labels):
    clf = LogisticRegression(max_iter=2000)
    clf.fit(train_embs.numpy(), train_labels.numpy())
    preds = clf.predict(test_embs.numpy())
    return accuracy_score(test_labels.numpy(), preds)

In [8]:
from sklearn.metrics import silhouette_score

def compute_silhouette(embs, labels):
    return silhouette_score(embs.numpy(), labels.numpy(), metric="cosine")

In [9]:
import torch.nn.functional as F

def embedding_separation_metrics(embs, labels):
    embs = F.normalize(embs, dim=1)
    sim_matrix = embs @ embs.T

    labels = labels.view(-1, 1)
    mask_same = labels == labels.T
    mask_diff = labels != labels.T

    intra = sim_matrix[mask_same].mean().item()
    inter = sim_matrix[mask_diff].mean().item()

    return intra, inter

In [10]:
def recall_at_k(embs, labels, k=5):
    embs = F.normalize(embs, dim=1)
    sim = embs @ embs.T

    correct = 0
    for i in range(len(embs)):
        sims = sim[i]
        sims[i] = -1  # ignore self
        topk = sims.topk(k).indices
        if (labels[topk] == labels[i]).any():
            correct += 1

    return correct / len(embs)

In [14]:
# Create plain training dataset and loader for evaluation
train_ds_plain = datasets.MNIST(
    root="./data", train=True, download=True,
    transform=transforms.ToTensor()
)
train_loader_plain = DataLoader(train_ds_plain, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)


train_embs, train_labels = extract_embeddings(model.encoder, train_loader_plain) # Use the new plain loader
test_embs, test_labels   = extract_embeddings(model.encoder, test_loader_plain) # This one is already plain

print("Embeddings extracted.")
print("Train shape:", train_embs.shape)
print("Test shape :", test_embs.shape)


# kNN Accuracy
knn_acc = knn_accuracy(train_embs, train_labels, test_embs, test_labels, k=10)

# Linear Probe Accuracy
linear_acc = linear_probe_accuracy(train_embs, train_labels, test_embs, test_labels)

# Silhouette Score (test set)
sil_score = compute_silhouette(test_embs, test_labels)

# Intra / Inter similarity
intra_sim, inter_sim = embedding_separation_metrics(test_embs, test_labels)
gap = intra_sim - inter_sim

# Recall@K
recall5  = recall_at_k(test_embs, test_labels, k=5)
recall10 = recall_at_k(test_embs, test_labels, k=10)

print("\n" + "="*50)
print("      EMBEDDING MODEL EVALUATION")
print("="*50)

print(f"kNN-10 Accuracy        : {knn_acc:.4f}")
print(f"Linear Probe Accuracy  : {linear_acc:.4f}")
print("-"*50)
print(f"Recall@5               : {recall5:.4f}")
print(f"Recall@10              : {recall10:.4f}")
print("-"*50)
print(f"Silhouette Score       : {sil_score:.4f}")
print("-"*50)
print(f"Intra-class similarity : {intra_sim:.4f}")
print(f"Inter-class similarity : {inter_sim:.4f}")
print(f"Similarity Gap         : {gap:.4f}")
print("="*50)


Embeddings extracted.
Train shape: torch.Size([60000, 128])
Test shape : torch.Size([10000, 128])

      EMBEDDING MODEL EVALUATION
kNN-10 Accuracy        : 0.9592
Linear Probe Accuracy  : 0.9631
--------------------------------------------------
Recall@5               : 0.9863
Recall@10              : 0.9923
--------------------------------------------------
Silhouette Score       : 0.2159
--------------------------------------------------
Intra-class similarity : 0.6369
Inter-class similarity : 0.3332
Similarity Gap         : 0.3036


In [17]:
model

RakEmbeddingModel(
  (encoder): EncoderCNN(
    (features): Sequential(
      (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): ReLU()
      (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (7): ReLU()
      (8): AdaptiveAvgPool2d(output_size=(1, 1))
    )
    (fc): Linear(in_features=128, out_features=128, bias=True)
  )
  (projector): ProjectionHead(
    (mlp): Sequential(
      (0): Linear(in_features=128, out_features=256, bias=True)
      (1): ReLU()
      (2): Linear(in_features=256, out_features=128, bias=True)
    )
  )
)